# Run 2 (1 year): true sky envelope, wavelet vs OG

1. Inject the true sky envelope $P_c(t) = M_c^2(t)$.
2. Simulate one 1-year series.
3. Wavelet sampler on it.
4. OG chunked pipeline on the same series (24 chunks of 15.21 d).
5. Compare the recovered $P_c(t)$ and spectral parameters.

Settings: [wavelet.yaml](../config/wavelet.yaml), [og_config.yaml](../config/og_config.yaml). Needs run 1 (reference
fit). Run the OG step from a terminal ([README](README.md)): nessai's worker pool deadlocks inside a notebook.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import corner

ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'simulations' / 'common'))
import injection as I
import diagnostics as D
from bahamas.wavelet.model import DAY
from bahamas.wavelet.simulate import power_modulation
import h5py
import og as OG
from bahamas.wavelet.simulate import ENVELOPE_PARAMS

RERUN = False
CFG = I.load_config(ROOT / 'simulations' / '1_year' / 'config' / 'wavelet.yaml')
TAG = 'run2'
OUT = Path(I.out_dir(CFG)) / TAG
CH = ['A', 'E']
amp_true, amp_ref = I.true_amp(), I.load_reference(I.reference_path(CFG))['amp']   # reference fit: run 1

## 1. Injected modulation

In [ ]:
t = np.linspace(0, 365 * DAY, 3000)
P_inj = np.stack([np.asarray(power_modulation(I.sky_params(), t, c)) for c in range(2)])
fig, ax = plt.subplots(figsize=(11, 3))
for i, c in enumerate(CH):
    ax.plot(t / DAY, P_inj[i], label=c)
ax.set_xlabel('time [days]'); ax.set_ylabel(r'$M_c^2(t)$'); ax.legend()
print('sky parameters:', I.sky_params())

## 2. Simulate

In [ ]:
data, inj = I.load_or_simulate('sky', TAG, CFG, rerun=RERUN)
like, post = I.build_posterior(data, CFG)
print(f'WDM: {data.Nt} x {data.Nf} pixels; likelihood: {len(like.times)} time blocks x {like.ngroups} frequency groups')

## 3. Wavelet sampler

In [ ]:
res_path = OUT / 'chains.npz'
if RERUN or not res_path.exists():
    I.run('sky', TAG, CFG)
r = dict(np.load(res_path, allow_pickle=True))
print(f"runtime {float(r['runtime_s']) / 60:.0f} min, acceptance {np.mean(r['acceptance_fraction']):.2f}")
D.report(r, post, np.array([60., 180., 300.]) * DAY);
for c in CH:
    K, p = r[f'mod_{c}_bayes_K'], r[f'mod_{c}_bayes_posterior']
    print(c, 'p(K):', ' '.join(f'{k}:{q:.2f}' for k, q in zip(K, p) if q > 0.01))

## 4. OG pipeline

Correct Gamma dof (`og/`) and the paper's (`og_legacy/`, `--legacy-dof`).

In [ ]:
ogs = {}
for name, legacy in [('OG (2 dof)', False), ('OG (paper dof)', True)]:
    path = OUT / OG.og_name(legacy) / 'result.hdf5'
    if not path.exists():
        print(f'{name}: missing, run python simulations/common/og.py --config simulations/1_year/config/wavelet.yaml '
              f'--tag run2{" --legacy-dof" if legacy else ""}')
        continue
    with h5py.File(path, 'r') as f:
        ogs[name] = f['posterior_samples'][()]
    print(f'{name}: {len(ogs[name])} samples')

## 5. Comparison

Galactic power relative to the true amplitude, $10^{\,\mathrm{amp}-\mathrm{amp}_{\rm true}} P_c(t)$; the truth is $M_c^2(t)$.

In [ ]:
td = r['modulation_times'] / DAY
P_true = np.stack([np.asarray(power_modulation(I.sky_params(), r['modulation_times'], c)) for c in range(2)])
bands = {'wavelet': 10 ** (amp_ref - amp_true) * r['modulation_draws']}
rng = np.random.default_rng(0)
for name, s in ogs.items():
    d = s[rng.choice(len(s), 500, replace=False)]
    bands[name] = np.array([10 ** (float(x['amp']) - amp_true) * np.stack(
        [np.asarray(power_modulation({k: float(x[k]) for k in ENVELOPE_PARAMS}, r['modulation_times'], c)) for c in range(2)])
        for x in d])

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for i, c in enumerate(CH):
    for name, B in bands.items():
        q = np.percentile(B[:, i], [5, 50, 95], axis=0)
        axes[i].fill_between(td, q[0], q[2], alpha=0.35, label=f'{name} 90%')
        print(f'{c} {name:15s} median 90% width {100 * np.median((q[2] - q[0]) / P_true[i]):.1f}% of P')
    axes[i].plot(td, P_true[i], 'k--', lw=1, label='injected')
    axes[i].set_ylabel(f'{c}: power / true amp')
axes[0].legend(ncol=4, fontsize=8); axes[1].set_xlabel('time [days]')
fig.savefig(OUT / 'modulation_og_vs_wavelet.png', dpi=150)

In [ ]:
# spectral parameters both pipelines share
shared = ['alpha', 'fknee', 'fr1', 'fr2', 'A', 'P']
names = [str(n).split(':')[-1] for n in r['psd_names']]
truth = {p['name']: p['injected'] for s in I.og_sources().values() for p in s}
fig = corner.corner(np.column_stack([r['psd_chain'][:, names.index(k)] for k in shared]), labels=shared, color='C0',
                    truths=[truth[k] for k in shared], plot_datapoints=False, levels=(0.5, 0.9))
if 'OG (2 dof)' in ogs:
    corner.corner(np.column_stack([ogs['OG (2 dof)'][k] for k in shared]), fig=fig, color='C1', plot_datapoints=False, levels=(0.5, 0.9))
fig.legend(handles=[plt.Line2D([], [], color='C0', label='wavelet'), plt.Line2D([], [], color='C1', label='OG (2 dof)')])
fig.savefig(OUT / 'corner_spectral_og_vs_wavelet.png', dpi=110);